# Desafio Final — Análise de Dados de Energia com API Pública

**Curso:** Ciência da Computação  
**Disciplina:** Soluções em Energias Renováveis e Sustentáveis

## Situação-problema

Uma equipe de planejamento energético precisa analisar o comportamento da carga elétrica de uma região atendida pelo Sistema Interligado Nacional (SIN).

Os dados serão obtidos diretamente de uma API pública do **Operador Nacional do Sistema Elétrico (ONS)**. A conexão com a API e a preparação inicial do JSON já estão fornecidas. A partir daí, sua equipe deverá construir o DataFrame, organizar os dados, criar recortes, calcular indicadores, produzir gráficos e elaborar um relatório técnico.

> Todos os códigos, resultados, gráficos e respostas devem permanecer neste mesmo Notebook.

## 1. Fonte dos dados

API pública de **Carga Verificada do ONS**:

- Portal: https://dados.ons.org.br/
- Conjunto de dados: https://dados.ons.org.br/dataset/carga-energia-verificada
- Dicionário de dados: Carga Verificada do ONS

Neste notebook será utilizada inicialmente a área **SP — São Paulo**, no período de **01/08/2025 a 07/08/2025**.

Os campos usados na análise seguem a estrutura da Carga Verificada do ONS: `cod_areacarga`, `dat_referencia`, `din_referenciautc`, `val_cargaglobal` e demais componentes disponibilizados pela API.


## 2. Bibliotecas

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

## 3. Consulta à API

Esta célula está pronta. Não é necessário conhecer `requests` para realizar o desafio.

In [ ]:
url = "https://apicarga.ons.org.br/prd/cargaverificada"

parametros = {
    "dat_inicio": "2025-08-01",
    "dat_fim": "2025-08-07",
    "cod_areacarga": "SP"
}

response = requests.get(url, params=parametros, timeout=30)

print("Status:", response.status_code)
print("URL:", response.url)

response.raise_for_status()
dados_json = response.json()

## 4. Preparação inicial do JSON

A célula abaixo localiza a lista principal de registros retornada pela API e a armazena em `registros`.

In [ ]:
if isinstance(dados_json, list):
    registros = dados_json

elif isinstance(dados_json, dict):
    chaves_com_lista = [
        chave for chave, valor in dados_json.items()
        if isinstance(valor, list)
    ]

    if not chaves_com_lista:
        raise ValueError("A resposta não contém uma lista de registros.")

    chave_registros = chaves_com_lista[0]
    registros = dados_json[chave_registros]
    print("Chave utilizada:", chave_registros)

else:
    raise TypeError("Formato de JSON não reconhecido.")

print("Tipo:", type(registros))
print("Quantidade de registros:", len(registros))

if registros:
    print("\nPrimeiro registro:")
    print(registros[0])

In [ ]:
type(registros)

In [ ]:
registros

# A partir daqui, o trabalho é da equipe

## DESAFIO 1 — Construção e inspeção do DataFrame

1. Crie um DataFrame Pandas chamado `dados` a partir de `registros`.
2. Exiba os primeiros registros.
3. Determine a quantidade de linhas e colunas.
4. Liste os nomes dos atributos.
5. Utilize `info()`.
6. Utilize `describe()`.
7. Em Markdown, identifique quais atributos representam data/hora, área de carga e valor de carga.

**Antes de prosseguir, compreenda a estrutura efetivamente retornada pela API.**

In [ ]:
# DESAFIO 1 — Construção e inspeção do DataFrame

# 1. Criar o DataFrame Pandas a partir de `registros`.
dados = pd.DataFrame(registros)

# 2. Exibir os primeiros registros.
print("# DESAFIO 1.2 — Primeiros registros")
display(dados.head())

# 3. Determinar a quantidade de linhas e colunas.
print("# DESAFIO 1.3 — Quantidade de linhas e colunas")
print("Linhas:", dados.shape[0])
print("Colunas:", dados.shape[1])

# 4. Listar os nomes dos atributos.
print("# DESAFIO 1.4 — Nomes dos atributos")
print(dados.columns.tolist())

# 5. Utilizar info().
print("# DESAFIO 1.5 — Informações do DataFrame")
dados.info()

# 6. Utilizar describe().
print("# DESAFIO 1.6 — Estatísticas descritivas")
display(dados.describe(include="all").T)

# 7. Identificação dos atributos solicitados.
# Na API atual de Carga Verificada do ONS, os campos relevantes são:
# cod_areacarga      -> código da área de carga
# din_referenciautc  -> data/hora de referência do final do intervalo
# val_cargaglobal    -> valor da carga global em MWmed
# Fonte: Dicionário de Dados — Carga Verificada (ONS).
col_area = "cod_areacarga"
col_data = "din_referenciautc"
col_carga = "val_cargaglobal"

campos_esperados = [col_area, col_data, col_carga]
campos_faltantes = [c for c in campos_esperados if c not in dados.columns]
if campos_faltantes:
    raise KeyError(
        "A resposta da API não contém os campos esperados: "
        + ", ".join(campos_faltantes)
        + ". Colunas recebidas: " + ", ".join(map(str, dados.columns))
    )

print("# DESAFIO 1.7 — Identificação dos atributos")
print(f"Data/hora: {col_data}")
print(f"Área de carga: {col_area}")
print(f"Valor de carga: {col_carga}")


## DESAFIO 1 — Identificação dos atributos

**# DESAFIO 1.7 — Resposta**

Na resposta da API de **Carga Verificada do ONS**, os atributos utilizados nesta análise são:

- **Data/hora:** `din_referenciautc`, que representa a data/hora de referência do final do intervalo da medição.
- **Área de carga:** `cod_areacarga`, que identifica a área de carga consultada; neste trabalho, o código utilizado na consulta é `SP`.
- **Valor de carga:** `val_cargaglobal`, que representa o valor da carga global em MWmed.

Esses campos correspondem ao dicionário oficial da API de Carga Verificada do ONS.


In [ ]:
# DESAFIO 2 — Organização dos dados

# Os campos foram identificados no DESAFIO 1 de acordo com a estrutura atual da API.

# 1. Renomear os principais atributos com nomes mais simples.
mapa_nomes = {
    col_data: "data_hora",
    col_area: "area",
    col_carga: "carga"
}

dados_organizados = dados.rename(columns=mapa_nomes).copy()

# 2. Criar um novo DataFrame contendo apenas os atributos necessários.
dados_analise = dados_organizados[["data_hora", "area", "carga"]].copy()

# 3. Verificar valores ausentes.
print("# DESAFIO 2.3 — Valores ausentes")
display(dados_analise.isna().sum().to_frame("quantidade_ausentes"))

# 4. Informar quantos valores ausentes existem em cada atributo relevante.
print("# DESAFIO 2.4 — Valores ausentes por atributo relevante")
for coluna in dados_analise.columns:
    print(f"{coluna}: {dados_analise[coluna].isna().sum()}")

# 5. Verificar se a variável de carga está em formato numérico.
print("# DESAFIO 2.5 — Tipo da variável de carga")
print("Tipo original:", dados_analise["carga"].dtype)
dados_analise["carga"] = pd.to_numeric(dados_analise["carga"], errors="coerce")
print("Tipo após conversão segura:", dados_analise["carga"].dtype)

# 6. Verificar como a data/hora está representada.
print("# DESAFIO 2.6 — Representação da data/hora")
print("Tipo original:", dados_analise["data_hora"].dtype)
dados_analise["data_hora"] = pd.to_datetime(dados_analise["data_hora"], errors="coerce", utc=True)
print("Tipo após conversão:", dados_analise["data_hora"].dtype)

# 7. Decisão de tratamento.
antes = len(dados_analise)
dados_analise = dados_analise.dropna(subset=["data_hora", "area", "carga"]).copy()
depois = len(dados_analise)

print("# DESAFIO 2.7 — Decisão de tratamento")
print(f"Foram removidas {antes - depois} linhas com ausência ou valor não conversível em data/hora, área ou carga.")
print("Nenhum valor de carga foi imputado; apenas registros sem os campos essenciais foram descartados.")

display(dados_analise.head())


## DESAFIO 2 — Decisão de tratamento

**# DESAFIO 2.7 — Resposta**

Os três atributos principais foram renomeados para `data_hora`, `area` e `carga`, e foi criado o DataFrame `dados_analise` contendo somente esses campos.

A variável `carga` foi convertida para formato numérico e `data_hora` para `datetime` com informação de UTC. Registros que ficaram sem data/hora, área ou carga após a conversão foram removidos. Não foi realizada imputação de valores, evitando criar informações que não estejam presentes nos dados da API.


In [ ]:
# DESAFIO 3 — Indicadores da carga elétrica

# Garantir ordenação temporal para as análises seguintes.
dados_analise = dados_analise.sort_values("data_hora").reset_index(drop=True)

# 1. Carga mínima.
carga_minima = dados_analise["carga"].min()

# 2. Carga máxima.
carga_maxima = dados_analise["carga"].max()

# 3. Carga média.
carga_media = dados_analise["carga"].mean()

# 4. Mediana.
mediana_carga = dados_analise["carga"].median()

# 5. Amplitude entre máximo e mínimo.
amplitude_carga = carga_maxima - carga_minima

# 6. Quantidade total de medições.
quantidade_medicoes = len(dados_analise)

indicadores = pd.DataFrame({
    "Indicador": [
        "Carga mínima", "Carga máxima", "Carga média",
        "Mediana", "Amplitude", "Quantidade total de medições"
    ],
    "Valor": [
        carga_minima, carga_maxima, carga_media,
        mediana_carga, amplitude_carga, quantidade_medicoes
    ]
})

print("# DESAFIO 3 — Indicadores calculados")
display(indicadores)

# Resposta à pergunta do desafio.
distancia_max_media = carga_maxima - carga_media
razao_max_media = carga_maxima / carga_media if carga_media else float("nan")

print("# DESAFIO 3 — Resposta")
print(
    f"O valor máximo foi {distancia_max_media:.2f} acima da média, "
    f"correspondendo a aproximadamente {razao_max_media:.2f} vezes a média."
)
print(
    "Assim, a distância entre máximo e média deve ser avaliada em conjunto com a mediana "
    "e a amplitude; os indicadores acima mostram objetivamente o quanto o pico se afasta "
    "do comportamento central observado."
)


## DESAFIO 4 — Períodos de alta demanda

Considere como **alta demanda** os registros com carga superior a **90% da carga máxima**.

1. Calcule o limiar.
2. Crie um novo DataFrame com os registros acima dele.
3. Conte os registros.
4. Calcule o percentual em relação ao total.
5. Identifique o maior valor de carga.
6. Identifique a data e o horário do pico, quando disponíveis.

Responda:

**Os períodos próximos ao pico representam uma parcela grande ou pequena do período analisado?**

In [ ]:
# DESAFIO 4 — Períodos de alta demanda

# 1. Calcular o limiar de alta demanda: 90% da carga máxima.
limiar_alta_demanda = 0.90 * carga_maxima

# 2. Criar DataFrame com registros acima do limiar.
alta_demanda = dados_analise[dados_analise["carga"] > limiar_alta_demanda].copy()

# 3. Contar registros.
quantidade_alta_demanda = len(alta_demanda)

# 4. Calcular percentual em relação ao total.
percentual_alta_demanda = (
    quantidade_alta_demanda / quantidade_medicoes * 100
    if quantidade_medicoes else 0
)

# 5. Identificar o maior valor de carga.
pico = dados_analise.loc[dados_analise["carga"].idxmax()]

# 6. Identificar data e horário do pico.
momento_pico = pico["data_hora"]

print("# DESAFIO 4.1 — Limiar")
print(f"Limiar de alta demanda (90% do máximo): {limiar_alta_demanda:.2f}")

print("# DESAFIO 4.2 — DataFrame de alta demanda")
display(alta_demanda)

print("# DESAFIO 4.3 e 4.4 — Quantidade e percentual")
print("Registros de alta demanda:", quantidade_alta_demanda)
print(f"Percentual do período: {percentual_alta_demanda:.2f}%")

print("# DESAFIO 4.5 e 4.6 — Pico")
print(f"Maior carga: {carga_maxima:.2f}")
print(f"Momento do pico: {momento_pico}")

print("# DESAFIO 4 — Resposta")
if percentual_alta_demanda < 10:
    interpretacao_alta = "uma parcela pequena"
elif percentual_alta_demanda < 30:
    interpretacao_alta = "uma parcela moderada"
else:
    interpretacao_alta = "uma parcela grande"

print(
    f"Os períodos próximos ao pico representam {interpretacao_alta} do período analisado, "
    f"pois {quantidade_alta_demanda} de {quantidade_medicoes} medições "
    f"({percentual_alta_demanda:.2f}%) ficaram acima de 90% da carga máxima."
)


## DESAFIO 4 — Interpretação

**# DESAFIO 4 — Resposta**

Foi considerado como alta demanda todo registro superior a 90% da carga máxima. A quantidade e o percentual encontrados são calculados diretamente na célula anterior.

A conclusão é baseada somente na proporção das medições que ultrapassaram o limiar, sem atribuir uma causa externa ao comportamento observado.


In [ ]:
# DESAFIO 5 — Segundo critério de análise

# 1. Critério escolhido:
# carga acima da média observada no período.
criterio_segundo = "Carga acima da média"

segundo_criterio = dados_analise[dados_analise["carga"] > carga_media].copy()

# 3. Quantidade de registros.
quantidade_segundo = len(segundo_criterio)

# 4. Percentual.
percentual_segundo = (
    quantidade_segundo / quantidade_medicoes * 100
    if quantidade_medicoes else 0
)

# 5. Comparação com o conjunto de alta demanda.
comparacao = pd.DataFrame({
    "Critério": ["Alta demanda (>90% do máximo)", "Segundo critério (> média)"],
    "Quantidade": [quantidade_alta_demanda, quantidade_segundo],
    "Percentual (%)": [percentual_alta_demanda, percentual_segundo]
})

print("# DESAFIO 5.1 — Critério escolhido")
print(criterio_segundo)

print("# DESAFIO 5.2 — Novo DataFrame")
display(segundo_criterio)

print("# DESAFIO 5.3 e 5.4 — Quantidade e percentual")
print("Quantidade:", quantidade_segundo)
print(f"Percentual: {percentual_segundo:.2f}%")

print("# DESAFIO 5.5 — Comparação com alta demanda")
display(comparacao)

print("# DESAFIO 5 — Interpretação")
print(
    f"O critério de alta demanda selecionou {percentual_alta_demanda:.2f}% das medições, "
    f"enquanto o critério de carga acima da média selecionou {percentual_segundo:.2f}%. "
    "Como o segundo critério é menos restritivo, ele tende a abranger mais observações."
)


## DESAFIO 5 — Critério e comparação

**# DESAFIO 5.1 — Critério escolhido:** carga acima da média do período.

Esse critério foi escolhido por permitir identificar todas as observações que ficaram acima do comportamento central calculado para o conjunto.

**# DESAFIO 5.5 — Comparação:** a célula anterior apresenta lado a lado a quantidade e o percentual do segundo critério e do critério de alta demanda (>90% do máximo).


In [ ]:
# DESAFIO 6 — Visualização

# 1. Gráfico do comportamento da carga ao longo do tempo.
plt.figure(figsize=(12, 5))
plt.plot(dados_analise["data_hora"], dados_analise["carga"])
plt.title("Carga elétrica ao longo do período — São Paulo")
plt.xlabel("Data e hora")
plt.ylabel("Carga")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("# DESAFIO 6.1 — Interpretação")
print(
    "O gráfico apresenta a variação da carga ao longo das observações do período. "
    "Os maiores valores podem ser comparados visualmente com a média e com o pico calculados."
)

# 2. Segundo gráfico: distribuição das cargas.
plt.figure(figsize=(8, 5))
plt.hist(dados_analise["carga"], bins=20)
plt.axvline(carga_media, linestyle="--", label=f"Média: {carga_media:.2f}")
plt.axvline(limiar_alta_demanda, linestyle="--", label=f"90% do máximo: {limiar_alta_demanda:.2f}")
plt.title("Distribuição das cargas observadas")
plt.xlabel("Carga")
plt.ylabel("Quantidade de medições")
plt.legend()
plt.tight_layout()
plt.show()

print("# DESAFIO 6.2 — Interpretação")
print(
    "O histograma mostra como as medições estão distribuídas. "
    "A linha da média permite localizar o comportamento central e a linha de 90% do máximo "
    "permite visualizar a faixa definida como alta demanda."
)


## DESAFIO 6 — Interpretações dos gráficos

**# DESAFIO 6 — Gráfico 1:** a série temporal permite observar diretamente como a carga variou durante o período analisado e localizar visualmente os maiores picos.

**# DESAFIO 6 — Gráfico 2:** o histograma mostra a distribuição das medições e permite comparar a concentração dos valores com a média e com o limiar de alta demanda.

As interpretações não atribuem causas às variações, pois essas causas não podem ser determinadas somente pelos dados analisados.


In [ ]:
# DESAFIO 7 — Síntese para o relatório

inicio_periodo = dados_analise["data_hora"].min()
fim_periodo = dados_analise["data_hora"].max()

resumo_resultados = f"""
Região analisada: São Paulo (SP)
Período analisado: {inicio_periodo} a {fim_periodo}
Quantidade de registros: {quantidade_medicoes}
Carga mínima: {carga_minima:.2f}
Carga máxima: {carga_maxima:.2f}
Carga média: {carga_media:.2f}
Mediana: {mediana_carga:.2f}
Amplitude: {amplitude_carga:.2f}
Limiar de alta demanda (90% do máximo): {limiar_alta_demanda:.2f}
Registros de alta demanda: {quantidade_alta_demanda}
Percentual de alta demanda: {percentual_alta_demanda:.2f}%
Momento do pico: {momento_pico}
Segundo critério: {criterio_segundo}
Registros acima da média: {quantidade_segundo}
Percentual acima da média: {percentual_segundo:.2f}%
"""

print("# DESAFIO 7 — resumo_resultados")
print(resumo_resultados)


## OPCIONAL: Relatório técnico com apoio do Gemini

**# Conforme solicitado, a geração do relatório pelo Gemini foi ignorada.**

O relatório técnico foi elaborado diretamente a partir dos resultados calculados pela equipe e inserido na seção **Relatório final**, sem uso de geração automática por IA.


In [ ]:
# # DESAFIO 8 — Geração por Gemini: ignorada conforme solicitado.


In [ ]:
# # DESAFIO 8 — Geração por Gemini: ignorada conforme solicitado.


## DESAFIO 8 — Geração do relatório

O relatório deve:

- usar os resultados calculados;
- apresentar os principais indicadores;
- destacar o pico e os períodos de alta demanda;
- comparar os dois critérios;
- não inventar causas;
- diferenciar observações de hipóteses;
- terminar com uma conclusão.

In [ ]:
# DESAFIO 8 — Relatório técnico sem Gemini

# O relatório final abaixo é construído exclusivamente com os resultados
# produzidos nas etapas anteriores. Nenhuma causa externa é inventada.
relatorio_final = f"""
RELATÓRIO TÉCNICO — ANÁLISE DA CARGA ELÉTRICA DE SÃO PAULO

1. Caracterização do conjunto analisado
Foi analisada a carga elétrica da área de carga de São Paulo (SP), utilizando os
dados de Carga Verificada do ONS para o período de {inicio_periodo} a {fim_periodo}.
O conjunto válido possui {quantidade_medicoes} medições.

2. Principais indicadores
A carga mínima observada foi de {carga_minima:.2f}, enquanto a carga máxima foi de
{carga_maxima:.2f}. A carga média foi de {carga_media:.2f} e a mediana foi de
{mediana_carga:.2f}. A amplitude entre o menor e o maior valor foi de
{amplitude_carga:.2f}.

O valor máximo ficou {distancia_max_media:.2f} acima da média. Portanto, os
indicadores mostram a distância do pico em relação ao comportamento médio, sem
atribuir uma causa para essa diferença.

3. Períodos de alta demanda
Foi definido como alta demanda todo registro superior a 90% da carga máxima.
O limiar calculado foi de {limiar_alta_demanda:.2f}. Foram identificadas
{quantidade_alta_demanda} medições nessa condição, correspondentes a
{percentual_alta_demanda:.2f}% do total.

O maior valor registrado foi de {carga_maxima:.2f}, com ocorrência em
{momento_pico}.

4. Comparação com o segundo critério
O segundo critério adotado foi carga acima da média. Esse recorte reuniu
{quantidade_segundo} medições, equivalentes a {percentual_segundo:.2f}% do total.
Em comparação, o critério de alta demanda selecionou {quantidade_alta_demanda}
medições ({percentual_alta_demanda:.2f}%).

Assim, o critério acima da média abrangeu uma parcela maior do conjunto do que o
critério de alta demanda, pois este último exige proximidade do valor máximo.

5. Conclusão
Os dados analisados mostram a variação da carga elétrica de São Paulo durante o
período estudado. O pico foi identificado objetivamente a partir do maior valor
de carga, e os períodos de alta demanda foram definidos pelo limite de 90% desse
máximo. O segundo recorte, baseado na média, permitiu uma visão mais abrangente
das observações acima do comportamento central.

As conclusões apresentadas são descritivas e baseadas exclusivamente nos dados
obtidos e nos indicadores calculados. Não é possível determinar, a partir deste
conjunto de dados isoladamente, as causas das variações observadas.
"""

print(relatorio_final)


In [ ]:
# DESAFIO 9 — Validação crítica

# 1. Os indicadores foram utilizados corretamente?
print("# DESAFIO 9.1")
print(
    "Sim. O relatório utiliza diretamente mínimo, máximo, média, mediana, amplitude, "
    "quantidade de medições, limiar de 90%, quantidade/percentual de alta demanda e "
    "o resultado do segundo critério."
)

# 2. Há alguma afirmação que não pode ser confirmada pelos dados?
print("\n# DESAFIO 9.2")
print(
    "Não foram incluídas causas externas ou explicações que não possam ser verificadas "
    "pelos dados. O relatório se limita a descrever os valores e comparações calculadas."
)

# 3. Houve interpretação exagerada ou causalidade não demonstrada?
print("\n# DESAFIO 9.3")
print(
    "Não. A interpretação foi mantida descritiva. O relatório não afirma que temperatura, "
    "atividade econômica, horário comercial ou qualquer outro fator causou as variações."
)

# 4. Que alterações a equipe realizou no texto?
print("\n# DESAFIO 9.4")
print(
    "Como a geração pelo Gemini foi ignorada, não houve revisão de um texto gerado pela IA. "
    "O relatório foi escrito diretamente com base nos resultados calculados no notebook, "
    "mantendo apenas conclusões sustentadas pelos dados."
)


## DESAFIO 9 — Validação crítica

**# DESAFIO 9.1 — Indicadores:** foram utilizados diretamente os indicadores calculados no notebook.

**# DESAFIO 9.2 — Afirmações não confirmadas:** não foram incluídas causas ou explicações externas que não possam ser demonstradas pelos dados.

**# DESAFIO 9.3 — Exagero/causalidade:** a análise foi mantida descritiva, sem atribuir causas às variações.

**# DESAFIO 9.4 — Alterações realizadas:** conforme solicitado, o Gemini não foi utilizado; portanto, o relatório final foi escrito diretamente a partir dos resultados calculados.


# RELATÓRIO FINAL

**# DESAFIO 8 — Relatório técnico final**

O relatório final abaixo é construído diretamente a partir dos resultados calculados nas etapas anteriores, conforme solicitado. A geração automática do relatório pelo Gemini foi ignorada.

A análise utiliza a área de carga **SP — São Paulo**, consultada na API de Carga Verificada do ONS para o período definido no notebook.


In [ ]:
# # RELATÓRIO FINAL — versão completa
from IPython.display import Markdown, display
display(Markdown(relatorio_final.replace('\n', '  \n')))


---

## Entrega

O notebook deve apresentar:

- consulta à API executada;
- DataFrame criado;
- inspeção e organização dos dados;
- indicadores;
- pelo menos dois DataFrames derivados por critérios;
- percentuais;
- pelo menos dois gráficos;
- interpretações;
- síntese para a IA;
- relatório com apoio do Gemini;
- validação crítica;
- versão final revisada.

**Entregue somente este Notebook (.ipynb), com todas as células executadas e os resultados visíveis.**